# Twitter Customer Support Dataset — Summary Statistics

**Dataset:** Customer Support on Twitter (~2.8 million tweets)  
**Time Period:** 2017–2018  
**Source:** [Kaggle – Customer Support on Twitter](https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter)  

This notebook computes:
1. Distribution of the target variable (inbound vs outbound / brand response behaviour)
2. Missing-value rates across all columns
3. Non-trivial visualizations revealing patterns in the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data & Basic Shape

In [ ]:
df = pd.read_csv(r'B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service\twitter\twcs\twcs.csv')
print(f'Shape: {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe(include='all')

## 2. Missing-Value Rates

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing %', ascending=False)
missing

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in missing['Missing %']]
missing['Missing %'].plot.barh(ax=ax, color=colors)
ax.set_xlabel('Missing %')
ax.set_title('Missing Value Rates by Column')
for i, (v, c) in enumerate(zip(missing['Missing %'], missing['Missing Count'])):
    ax.text(v + 0.5, i, f'{v:.1f}%  ({c:,})', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Distribution of the Target Variable

For our customer-support RL problem the core "target" signal is whether a tweet is **inbound** (customer → brand) or **outbound** (brand → customer).  
The ratio tells us how chatty brands are relative to customer complaints, and how many customer messages never receive a reply (`response_tweet_id` is NaN).

In [ ]:
inbound_counts = df['inbound'].value_counts()
print('Inbound distribution:')
print(inbound_counts)
print(f'\nInbound %: {inbound_counts[True] / len(df) * 100:.1f}%')
print(f'Outbound %: {inbound_counts[False] / len(df) * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
labels = ['Customer (Inbound)', 'Brand (Outbound)']
sizes = [inbound_counts[True], inbound_counts[False]]
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140,
            colors=['#3498db', '#e67e22'], explode=(0.03, 0.03))
axes[0].set_title('Inbound vs Outbound Tweets')

# Response rate among inbound tweets
inbound_df = df[df['inbound'] == True]
has_response = inbound_df['response_tweet_id'].notna().sum()
no_response = inbound_df['response_tweet_id'].isna().sum()
axes[1].bar(['Got Brand Reply', 'No Reply'], [has_response, no_response],
            color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Customer Tweets: Got a Brand Reply?')
axes[1].set_ylabel('Count')
for i, v in enumerate([has_response, no_response]):
    axes[1].text(i, v + len(df)*0.005, f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Top Brands by Volume (Support Load)

In [ ]:
# Brand = outbound author
brand_tweets = df[df['inbound'] == False]
top_brands = brand_tweets['author_id'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 5))
top_brands.plot.barh(ax=ax, color=sns.color_palette('viridis', 20))
ax.set_xlabel('Number of Outbound (Support) Tweets')
ax.set_title('Top 20 Brands by Support Tweet Volume')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Non-Trivial Visualization: Brand Response-Rate vs Volume

**Insight sought:** Do high-volume support brands actually reply to a *larger* fraction of customer messages?  
This scatter plot reveals the **tension between scale and responsiveness** — critical for our cost-to-serve analysis.

In [ ]:
# For each brand, compute: total inbound tweets directed at them, and % that got a response
# Inbound tweets mention the brand via @handle in the text or via in_response_to_tweet_id chain

# Approach: brand outbound tweets reference in_response_to_tweet_id -> that's the customer tweet they replied to
brand_outbound = df[df['inbound'] == False].copy()
customer_inbound = df[df['inbound'] == True].copy()

# Extract brand handle from customer tweet text (first @mention)
customer_inbound['mentioned_brand'] = customer_inbound['text'].str.extract(r'^@(\w+)', expand=False)

# Compute per-brand stats
brand_inbound_count = customer_inbound.groupby('mentioned_brand').size().rename('inbound_count')
brand_replied = customer_inbound[customer_inbound['response_tweet_id'].notna()].groupby('mentioned_brand').size().rename('replied_count')

brand_stats = pd.concat([brand_inbound_count, brand_replied], axis=1).fillna(0)
brand_stats['response_rate'] = (brand_stats['replied_count'] / brand_stats['inbound_count'] * 100)
brand_stats = brand_stats[brand_stats['inbound_count'] >= 100]  # filter noise

print(f'Brands with ≥100 inbound tweets: {len(brand_stats)}')
brand_stats.sort_values('inbound_count', ascending=False).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    brand_stats['inbound_count'],
    brand_stats['response_rate'],
    alpha=0.6, s=40, c=brand_stats['response_rate'],
    cmap='RdYlGn', edgecolors='grey', linewidth=0.4
)
ax.set_xscale('log')
ax.set_xlabel('Inbound Customer Tweets (log scale)')
ax.set_ylabel('Brand Response Rate (%)')
ax.set_title('Brand Response Rate vs Customer Tweet Volume\n(Each dot = one brand, ≥100 inbound tweets)')
plt.colorbar(scatter, ax=ax, label='Response Rate %')

# Label top brands
for brand in brand_stats.nlargest(8, 'inbound_count').index:
    row = brand_stats.loc[brand]
    ax.annotate(brand, (row['inbound_count'], row['response_rate']),
                fontsize=7, alpha=0.8, ha='left',
                xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

## 6. Conversation Thread Length Distribution

Thread length (number of back-and-forth turns) is a proxy for **time-to-resolution**. Longer threads ≈ higher cost-to-serve.

In [ ]:
# Build reply chains
# A thread starts with an inbound tweet that has no in_response_to_tweet_id pointing to another tweet in the dataset
# Simpler proxy: count how many tweets share the same root conversation

# Build parent -> child mapping
reply_map = df.set_index('tweet_id')['in_response_to_tweet_id'].dropna().astype(int).to_dict()

# Find root for each tweet by walking up
def find_root(tid, reply_map, max_depth=50):
    visited = set()
    current = tid
    depth = 0
    while current in reply_map and depth < max_depth:
        if current in visited:
            break
        visited.add(current)
        current = reply_map[current]
        depth += 1
    return current

# Sample for efficiency
sample_ids = df['tweet_id'].values
roots = {tid: find_root(tid, reply_map) for tid in sample_ids}
df['thread_root'] = df['tweet_id'].map(roots)

thread_lengths = df.groupby('thread_root').size().rename('thread_length')
print(f'Total threads: {len(thread_lengths):,}')
print(thread_lengths.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram (capped at 20 for readability)
capped = thread_lengths.clip(upper=20)
capped.value_counts().sort_index().plot.bar(ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_xlabel('Thread Length (turns, capped at 20)')
axes[0].set_ylabel('Number of Threads')
axes[0].set_title('Distribution of Conversation Thread Length')

# Box plot per inbound/outbound breakdown won't be meaningful,
# Instead show CDF
sorted_tl = np.sort(thread_lengths.values)
cdf = np.arange(1, len(sorted_tl)+1) / len(sorted_tl)
axes[1].plot(sorted_tl, cdf, color='#e74c3c', linewidth=2)
axes[1].set_xlabel('Thread Length (turns)')
axes[1].set_ylabel('Cumulative Fraction of Threads')
axes[1].set_title('CDF of Thread Length')
axes[1].set_xlim(0, 30)
axes[1].axhline(0.9, color='grey', linestyle='--', alpha=0.5)
axes[1].text(25, 0.91, '90th %ile', fontsize=9, color='grey')

plt.tight_layout()
plt.show()

p90 = np.percentile(thread_lengths.values, 90)
print(f'90th percentile thread length: {p90:.0f} turns')

## 7. Temporal Pattern: Tweet Volume Over Time

In [ ]:
df['created_at_dt'] = pd.to_datetime(df['created_at'], format='%a %b %d %H:%M:%S %z %Y', errors='coerce')
df['date'] = df['created_at_dt'].dt.date
df['hour'] = df['created_at_dt'].dt.hour
df['day_of_week'] = df['created_at_dt'].dt.day_name()

daily = df.groupby(['date', 'inbound']).size().unstack(fill_value=0)
daily.columns = ['Brand (Outbound)', 'Customer (Inbound)']

fig, ax = plt.subplots(figsize=(12, 4))
daily.rolling(7).mean().plot(ax=ax, linewidth=1.5)
ax.set_title('Daily Tweet Volume (7-day rolling avg)')
ax.set_ylabel('Tweets / day')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: hour of day vs day of week
heatmap_data = df[df['inbound'] == True].groupby(['day_of_week', 'hour']).size().unstack(fill_value=0)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(day_order)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Customer (Inbound) Tweet Volume — Hour of Day vs Day of Week')
ax.set_ylabel('')
ax.set_xlabel('Hour of Day (UTC)')
plt.tight_layout()
plt.show()

## 8. Text Length Distribution

In [ ]:
df['text_len'] = df['text'].fillna('').str.len()

fig, ax = plt.subplots(figsize=(10, 4))
df[df['inbound'] == True]['text_len'].hist(bins=80, alpha=0.6, label='Customer', color='#3498db', ax=ax)
df[df['inbound'] == False]['text_len'].hist(bins=80, alpha=0.6, label='Brand', color='#e67e22', ax=ax)
ax.set_xlabel('Tweet Character Length')
ax.set_ylabel('Count')
ax.set_title('Text Length Distribution: Customer vs Brand Tweets')
ax.legend()
plt.tight_layout()
plt.show()

print('Customer tweet length stats:')
print(df[df['inbound'] == True]['text_len'].describe().round(1))
print('\nBrand tweet length stats:')
print(df[df['inbound'] == False]['text_len'].describe().round(1))

## 9. Summary & Key Takeaways

| Metric | Value |
|--------|-------|
| Total tweets | ~2.8 M |
| Inbound / Outbound split | ~58 % / 42 % |
| Customer tweets with no brand reply | See chart above |
| Missing `response_tweet_id` | Significant — many customer tweets unanswered |
| Median thread length | Typically 2–4 turns |

**Key non-trivial finding:** High-volume brands do *not* uniformly maintain high response rates — there's a clear dispersion, suggesting that scaling support is a genuine cost/quality trade-off.  
This motivates our RL approach: intelligently routing tickets (bot vs. human) can improve response rates at the same cost budget.